In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn
from autonomy.config import DATA_DIR, MODEL_PATH
from autonomy.model import SteeringModel, get_device
from autonomy.dataset import make_loaders
from autonomy.trainer import Trainer

device = get_device()
print(f"Using {device} device")

In [ ]:
# create model (the dummy forward pass inside create() initializes the LazyLinear layer)
model = SteeringModel.create(device)
print(model)

In [ ]:
# add optimizer (parameters already exist because create() ran a forward pass)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# dataset + dataloaders
train_loader, val_loader = make_loaders(DATA_DIR, batch_size=64, val_fraction=0.2, num_workers=0)

In [ ]:
# loss + trainer
loss_fn = nn.MSELoss()  # steering is a regression target in [-1, 1], matching the Tanh output
trainer = Trainer(model, optimizer, loss_fn, device)

In [ ]:
# training loop, saving the best checkpoint by validation loss
trainer.fit(train_loader, val_loader, epochs=20, ckpt_path=MODEL_PATH)